# GraviTree on COMPAS and FICO

Fit a Rashomon set of falling trees on COMPAS and FICO. `binarize=True` turns continuous features into threshold features during `fit`. 


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

from falling_trees.gravitree import GraviTree

ROOT = Path.cwd()
if not (ROOT / "data" / "benchmark" / "compas.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "benchmark"

def load_split(csv_name, label, test_size=0.2, seed=0):
    frame = pd.read_csv(DATA / csv_name)
    train, test = train_test_split(
        frame, test_size=test_size, random_state=seed, stratify=frame[label]
    )
    return (
        train.drop(columns=[label]),
        train[label].astype(int),
        test.drop(columns=[label]),
        test[label].astype(int),
    )

def fit_and_print(csv_name, label, dataset_name):
    X_train, y_train, X_test, y_test = load_split(csv_name, label)
    model = GraviTree(
        depth=5,
        eps=0.02,
        min_support=0.05,
        branching_cost=0.05,
        binarize=True,
        num_estimators=50,
    )
    model.fit(X_train, y_train)
    scores = [model.score(X_train, y_train, tree_idx=i) for i in range(len(model.ft_rset))] # scores are ordered by accuracy
    index = max(range(len(scores)), key=scores.__getitem__)
    test_acc = model.score(X_test, y_test, tree_idx=index)
    print(
        f"{dataset_name}: {len(model.ft_rset)} trees, "
        f"{len(model.feature_names_)} binary features, "
        f"{len(y_train)} train / {len(y_test)} test rows"
    )
    print(
        f"Most accurate tree is index {index}: "
        f"train accuracy {scores[index]:.3f}, test accuracy {test_acc:.3f}"
    )
    model.print_tree(tree_idx=index)
    return model



## COMPAS

In [2]:
compas = fit_and_print('compas.csv', 'two_year_recid', 'COMPAS')

COMPAS: 360 trees, 25 binary features, 5525 train / 1382 test rows
Most accurate tree is index 0: train accuracy 0.666, test accuracy 0.651
if juvenile_crimes<=0.5:
    if priors_count<=4.5:
        if age<=21.5:
            return 0.5776173285198556
        else:
            if priors_count<=2.5:
                if age<=27.5:
                    return 0.3877551020408163
                else:
                    return 0.2607124419204956
            else:
                return 0.5139573070607554
    else:
        return 0.6502890173410405
else:
    return 0.7186700767263428


## FICO

In [3]:
fico = fit_and_print('fico.csv', 'RiskPerformance', 'FICO')

FICO: 64 trees, 27 binary features, 8367 train / 2092 test rows
Most accurate tree is index 0: train accuracy 0.699, test accuracy 0.696
if ExternalRiskEstimate<=72.5:
    return 0.30137575236457437
else:
    return 0.6993270524899058
